# Semana 7: Formas y forward propagation

**Pregunta de trabajo.** Rastrear una red 64-h-10, demostrar el colapso afín y probar softmax estable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

## Dataset local
Usamos `load_digits`, que viene con scikit-learn y no requiere internet.

In [2]:
digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
X_train.shape, X_val.shape

((1347, 64), (450, 64))

## Funciones de forward

In [3]:
def one_hot(y, num_classes):
    y = np.asarray(y, dtype=int)
    out = np.zeros((len(y), num_classes), dtype=float)
    out[np.arange(len(y)), y] = 1.0
    return out


def relu(Z):
    return np.maximum(0.0, Z)


def relu_derivative(Z):
    return (Z > 0).astype(float)


def softmax(Z):
    Z = np.asarray(Z, dtype=float)
    shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def cross_entropy_loss(P, Y, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    return float(-np.sum(Y * np.log(P)) / len(Y))


def init_parameters(n_features, n_hidden, n_classes, seed=42):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0.0, np.sqrt(2.0 / n_features), size=(n_features, n_hidden))
    b1 = np.zeros(n_hidden)
    W2 = rng.normal(0.0, np.sqrt(2.0 / n_hidden), size=(n_hidden, n_classes))
    b2 = np.zeros(n_classes)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward_pass(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    P = softmax(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "P": P}
    return P, cache

In [4]:
params = init_parameters(X_train.shape[1], 32, 10, seed=7)
P, cache = forward_pass(X_train[:5], params)
P.shape, P.sum(axis=1)

((5, 10), array([1., 1., 1., 1., 1.]))

## Pregunta de cierre
¿Qué forma tendría `W2` si cambiamos a 20 neuronas ocultas y 10 clases?

## Pruebas adicionales de forward

Se comprueban baseline, colapso de capas afines, estabilidad y escala inicial de la pérdida.

In [5]:
class_counts = pd.Series(y_train).value_counts().sort_index()
baseline_accuracy = class_counts.max() / class_counts.sum()
{"clases": class_counts.to_dict(), "accuracy_baseline": float(baseline_accuracy)}

{'clases': {0: 133,
  1: 136,
  2: 133,
  3: 137,
  4: 136,
  5: 136,
  6: 136,
  7: 134,
  8: 131,
  9: 135},
 'accuracy_baseline': 0.10170749814402376}

In [6]:
rng = np.random.default_rng(2105)
X_toy = rng.normal(size=(5, 4))
W1 = rng.normal(size=(4, 3)); b1 = rng.normal(size=3)
W2 = rng.normal(size=(3, 2)); b2 = rng.normal(size=2)
stacked = (X_toy @ W1 + b1) @ W2 + b2
collapsed = X_toy @ (W1 @ W2) + (b1 @ W2 + b2)
float(np.max(np.abs(stacked - collapsed)))

1.7763568394002505e-15

In [7]:
extreme = np.array([[1000.0, 1001.0, 999.0], [-1000.0, -999.0, -1001.0]])
P_extreme = softmax(extreme)
{"finite": bool(np.isfinite(P_extreme).all()), "row_sums": P_extreme.sum(axis=1)}

{'finite': True, 'row_sums': array([1., 1.])}

In [8]:
Y_batch = one_hot(y_train[:32], 10)
P_batch, cache_batch = forward_pass(X_train[:32], params)
initial_loss = cross_entropy_loss(P_batch, Y_batch)
stats = {
    "loss": initial_loss,
    "log_10": float(np.log(10)),
    "relu_inactive": float((cache_batch["A1"] == 0).mean()),
}
stats

{'loss': 2.7609679672591647,
 'log_10': 2.302585092994046,
 'relu_inactive': 0.5771484375}